# Exploratory — MLPs on raw ISIs, ISI features, sleep-architecture features, and all combined

| | |
|---|---|
| **Original** | `Liam_work/tests/behavior_isi_analysis/AD_Behavior_ISI_MLP_Classification.ipynb` |
| **Status** | Early exploration — **superseded** by the step 2 / step 3 notebooks. Kept for the record; uses older conventions (212-day cutoff, leave-one-animal-out, no hyperparameter search, `caf104` not excluded). (Mar 2026). The direct ancestor of step 3. |

Four MLP approaches, each on ALL / OLD / YOUNG with leave-one-animal-out CV: (1) raw ISI histograms
(sum-normalised here; step 3 later switched to min–max), (2) ~80 hand-crafted ISI features per state
(moments, quantiles, burst/pause fractions, log-ISI stats, entropy, epoch-to-epoch dynamics),
(3) sleep-architecture / "behavioural" features (state fractions, epoch spike counts, state-rate ratios,
recording phase), (4) all combined. Also a head-to-head comparison and a brain-region breakdown.

Caveat: the sleep-architecture features are derived from per-neuron epoch counts and are largely
animal-level properties, so they can identify animals rather than disease.

Paths: raw inputs are read from `/media/HlabShare/...` (read-only). Anything this copy writes goes under `/hlabhome/wg-mjames/git/LDCODES12_ind_neuron_spiking_classification_project/` — never back into `Liam_work/`.

---
*Repo copy: outputs stripped, header/section notes added, lab preamble added where missing, and output
paths moved into this repo. Analysis logic is unchanged from Liam's version.*

In [ ]:
# =============================================================================
# LAB-WIDE PREAMBLE — CPU thread limits + figure-export font config
# (added in the repo copy; set BEFORE numpy/scipy/sklearn/xgboost are imported)
# =============================================================================
import os
import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams.update({'font.family': 'DejaVu Sans'})

# Limit CPU threads — shared server "marvin"
os.environ['OMP_NUM_THREADS']        = '4'
os.environ['OPENBLAS_NUM_THREADS']   = '4'
os.environ['MKL_NUM_THREADS']        = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS']    = '4'

# AD vs WT Classification: ISI & Behavioral Feature Analysis
## MLP-Based Neural Classification Pipeline

### Strategy
1. **Part 1: Raw ISI MLP** — Feed raw ISI distributions (as histograms) directly to the MLP
2. **Part 2: ISI Feature MLP** — Classify using hand-crafted ISI distribution features
3. **Part 3: Behavioral Feature MLP** — Classify using sleep architecture & behavioral features
4. **Part 4: Full-Dimensional MLP** — Combine all features for maximum classification power

### Each approach runs 3 tests:
- **All animals** (20 animals, LOAO CV)
- **OLD only** (>212 days): WT vs AD with known phenotype
- **YOUNG only** (≤212 days): WT vs AD pre-symptomatic detection

### Dataset
- `all_animal_isi_df.pkl`: 5,501 neurons from 20 animals (11 WT, 9 AD)
- Age cutoff: 212 days (APP/PS1 symptom onset)
- OLD: 6 WT + 4 AD = 10 animals | YOUNG: 5 WT + 5 AD = 10 animals

In [ ]:
# =============================================================================
# SETUP AND DATA LOADING
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             confusion_matrix, roc_curve)
from sklearn.inspection import permutation_importance
from joblib import Parallel, delayed
import time
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-whitegrid')
np.random.seed(42)

AGE_CUTOFF = 212  # days — APP/PS1 symptom onset
N_JOBS = 20       # use 20 of 104 cores

# Load data
print('Loading dataset (this takes a moment - 13 GB file)...')
t0 = time.time()
df = pd.read_pickle('/media/HlabShare/Ben_work/isi/all_animal_isi_df.pkl')
df['age_group'] = df['age'].apply(lambda x: 'OLD' if x > AGE_CUTOFF else 'YOUNG')
df['geno_label'] = df['geno'].map({'wt': 'WT', 'app_ps1': 'AD'})
df['group'] = df['geno_label'] + '_' + df['age_group']
df['label'] = (df['geno'] == 'app_ps1').astype(int)

print('Dataset loaded in {:.1f}s.'.format(time.time() - t0))
print('Total neurons: {:,}'.format(len(df)))
print('Animals: {}'.format(df['animal'].nunique()))
print('Age cutoff: {} days | Using {} parallel cores'.format(AGE_CUTOFF, N_JOBS))
print()
for grp in ['WT_YOUNG', 'AD_YOUNG', 'WT_OLD', 'AD_OLD']:
    sub = df[df['group'] == grp]
    print('  {}: {} animals, {:,} neurons, ages {:.0f}-{:.0f}d'.format(
        grp, sub['animal'].nunique(), len(sub),
        sub['age'].min(), sub['age'].max()))


In [ ]:
# =============================================================================
# SHARED UTILITIES (PARALLELIZED)
# =============================================================================

meta_cols = ['animal', 'geno_label', 'age', 'age_group', 'brain_region',
             'cell_type', 'peak_latency', 'label']


def clean_isi_array(isi_list, min_isi=0, max_isi=60):
    """Concatenate and clean ISI values from epoch list."""
    all_isis = []
    for arr in isi_list:
        if isinstance(arr, np.ndarray) and len(arr) > 0:
            valid = arr[(arr > min_isi) & (arr <= max_isi)]
            all_isis.extend(valid)
    return np.array(all_isis)


def _train_one_fold(train_idx, test_idx, X, y, groups, genos, hidden_layers):
    """Train one LOAO fold — designed to run in parallel."""
    test_animal = groups[test_idx][0]
    test_geno = genos[test_idx][0]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_idx])
    X_test = scaler.transform(X[test_idx])

    mlp = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation='relu', solver='adam',
        alpha=0.01, batch_size=256,
        learning_rate='adaptive', learning_rate_init=0.001,
        max_iter=300, early_stopping=True,
        validation_fraction=0.15, n_iter_no_change=10,
        random_state=42
    )
    mlp.fit(X_train, y[train_idx])

    preds = mlp.predict(X_test)
    probs = mlp.predict_proba(X_test)[:, 1]

    neuron_acc = accuracy_score(y[test_idx], preds)
    animal_pred = 1 if np.mean(probs) > 0.5 else 0
    animal_true = y[test_idx][0]

    return {
        'test_idx': test_idx,
        'preds': preds,
        'probs': probs,
        'animal': test_animal,
        'geno': test_geno,
        'true': animal_true,
        'pred': animal_pred,
        'mean_prob': np.mean(probs),
        'neuron_acc': neuron_acc,
        'n_neurons': len(test_idx)
    }


def run_mlp_classification(feature_df, feat_cols, group_col='animal', label_col='label',
                           title='MLP Classification', hidden_layers=(128, 64, 32)):
    """
    Train and evaluate an MLP with Leave-One-Animal-Out CV.
    All folds run in parallel using joblib.
    """
    work_df = feature_df.copy()
    X = work_df[feat_cols].values.astype(float)
    y = work_df[label_col].values
    groups = work_df[group_col].values
    genos = work_df['geno_label'].values

    # Fill NaNs with column medians
    for col_idx in range(X.shape[1]):
        nan_mask = np.isnan(X[:, col_idx])
        if nan_mask.any():
            median_val = np.nanmedian(X[:, col_idx])
            if np.isnan(median_val):
                median_val = 0
            X[nan_mask, col_idx] = median_val

    print('  Data: {:,} neurons, {} features'.format(len(X), X.shape[1]))
    print('  Class balance: {} AD, {} WT'.format(sum(y == 1), sum(y == 0)))

    logo = LeaveOneGroupOut()
    folds = list(logo.split(X, y, groups))
    n_folds = len(folds)
    print('  Running {} LOAO folds in parallel (n_jobs={})...'.format(n_folds, N_JOBS))

    t0 = time.time()
    fold_results = Parallel(n_jobs=N_JOBS)(
        delayed(_train_one_fold)(train_idx, test_idx, X, y, groups, genos, hidden_layers)
        for train_idx, test_idx in folds
    )
    elapsed = time.time() - t0
    print('  {} folds completed in {:.1f}s'.format(n_folds, elapsed))

    # Reassemble results
    all_preds = np.zeros(len(y))
    all_probs = np.zeros(len(y))
    animal_results = []

    for fr in fold_results:
        all_preds[fr['test_idx']] = fr['preds']
        all_probs[fr['test_idx']] = fr['probs']
        animal_results.append({
            'animal': fr['animal'], 'geno': fr['geno'],
            'true': fr['true'], 'pred': fr['pred'],
            'mean_prob': fr['mean_prob'], 'neuron_acc': fr['neuron_acc'],
            'n_neurons': fr['n_neurons']
        })

    animal_res_df = pd.DataFrame(animal_results)
    animal_acc = accuracy_score(animal_res_df['true'], animal_res_df['pred'])
    animal_auc = roc_auc_score(animal_res_df['true'], animal_res_df['mean_prob'])
    neuron_acc = accuracy_score(y, all_preds)
    neuron_auc = roc_auc_score(y, all_probs)

    print('  --- {} ---'.format(title))
    print('  NEURON:  Acc={:.1%}  AUC={:.3f}'.format(neuron_acc, neuron_auc))
    print('  ANIMAL:  Acc={:.1%} ({}/{})  AUC={:.3f}'.format(
        animal_acc, int(sum(animal_res_df['true'] == animal_res_df['pred'])),
        len(animal_res_df), animal_auc))

    for _, r in animal_res_df.sort_values('geno').iterrows():
        mark = 'Y' if r['true'] == r['pred'] else 'X'
        print('    {} ({}) prob={:.3f} nacc={:.1%} [{}]'.format(
            r['animal'], r['geno'], r['mean_prob'], r['neuron_acc'], mark))

    return {
        'animal_results': animal_res_df,
        'animal_acc': animal_acc, 'animal_auc': animal_auc,
        'neuron_acc': neuron_acc, 'neuron_auc': neuron_auc,
        'y_true': y, 'y_probs': all_probs, 'y_preds': all_preds,
        'title': title
    }


def run_all_three_splits(feature_df, feat_cols, approach_name, hidden_layers=(128, 64, 32)):
    """Run classification on ALL, OLD-only, and YOUNG-only splits."""
    results = {}

    print('\n' + '=' * 70)
    print('{} — ALL ANIMALS'.format(approach_name))
    print('=' * 70)
    results['all'] = run_mlp_classification(
        feature_df, feat_cols,
        title='{} (All)'.format(approach_name),
        hidden_layers=hidden_layers
    )

    for age_grp in ['OLD', 'YOUNG']:
        print('\n' + '=' * 70)
        print('{} — {} ONLY (>212d)'.format(approach_name, age_grp) if age_grp == 'OLD'
              else '{} — {} ONLY (<=212d)'.format(approach_name, age_grp))
        print('=' * 70)
        subset = feature_df[feature_df['age_group'] == age_grp].copy()
        n_animals = subset['animal'].nunique()
        n_ad = subset[subset['label'] == 1]['animal'].nunique()
        n_wt = subset[subset['label'] == 0]['animal'].nunique()
        print('  {} animals: {} WT, {} AD'.format(n_animals, n_wt, n_ad))
        if n_ad < 2 or n_wt < 2:
            print('  SKIPPED: need at least 2 animals per class')
            continue
        results[age_grp.lower()] = run_mlp_classification(
            subset, feat_cols,
            title='{} ({})'.format(approach_name, age_grp),
            hidden_layers=hidden_layers
        )

    return results


def plot_results_grid(results_dict, suptitle):
    """Plot ROC + confusion matrix + per-animal probs for a set of results."""
    splits = [k for k in ['all', 'old', 'young'] if k in results_dict]
    n_splits = len(splits)
    fig, axes = plt.subplots(n_splits, 3, figsize=(16, 5 * n_splits))
    if n_splits == 1:
        axes = axes.reshape(1, -1)

    split_labels = {'all': 'All Animals', 'old': 'OLD Only (>212d)', 'young': 'YOUNG Only (<=212d)'}

    for row, split in enumerate(splits):
        res = results_dict[split]

        # ROC
        ax = axes[row, 0]
        fpr, tpr, _ = roc_curve(res['y_true'], res['y_probs'])
        ax.plot(fpr, tpr, 'b-', lw=2, label='Neuron AUC={:.3f}'.format(res['neuron_auc']))
        ar = res['animal_results']
        fpr_a, tpr_a, _ = roc_curve(ar['true'], ar['mean_prob'])
        ax.plot(fpr_a, tpr_a, 'r-', lw=2, label='Animal AUC={:.3f}'.format(res['animal_auc']))
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
        ax.set_xlabel('FPR')
        ax.set_ylabel('TPR')
        ax.set_title('{}: ROC'.format(split_labels[split]), fontweight='bold')
        ax.legend(fontsize=9)

        # Confusion matrix
        ax = axes[row, 1]
        cm = confusion_matrix(res['y_true'], res['y_preds'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['WT', 'AD'], yticklabels=['WT', 'AD'])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title('{}: Confusion'.format(split_labels[split]))

        # Per-animal probability
        ax = axes[row, 2]
        ar_s = ar.sort_values('mean_prob')
        colors = ['crimson' if g == 'AD' else 'steelblue' for g in ar_s['geno']]
        ax.barh(range(len(ar_s)), ar_s['mean_prob'], color=colors, edgecolor='black')
        ax.set_yticks(range(len(ar_s)))
        ax.set_yticklabels(['{} ({})'.format(a, g) for a, g in zip(ar_s['animal'], ar_s['geno'])])
        ax.axvline(x=0.5, color='black', linestyle='--', lw=1.5)
        ax.set_xlabel('P(AD)')
        ax.set_title('{}: Per-Animal'.format(split_labels[split]))

    fig.suptitle(suptitle, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Store all results for final comparison
master_results = {}
print('Utilities defined. Using {} parallel cores.'.format(N_JOBS))


---
# Part 1: Raw ISI MLP
Feed the actual ISI distributions directly to the MLP as log-spaced histogram bins.
No hand-crafted features — let the network learn what matters.

In [ ]:
# =============================================================================
# RAW ISI HISTOGRAM REPRESENTATION (PARALLELIZED)
# =============================================================================
print('=' * 70)
print('PART 1: RAW ISI — BUILDING HISTOGRAM REPRESENTATION')
print('=' * 70)

N_BINS = 100
BIN_EDGES = np.logspace(np.log10(0.001), np.log10(60), N_BINS + 1)


def isi_to_histogram(isi_list, bin_edges=BIN_EDGES):
    """Convert raw ISI arrays to a normalized histogram."""
    isis = clean_isi_array(isi_list)
    if len(isis) < 10:
        return np.zeros(len(bin_edges) - 1)
    hist, _ = np.histogram(isis, bins=bin_edges)
    total = hist.sum()
    if total > 0:
        hist = hist.astype(float) / total
    return hist


def _process_raw_isi_row(row):
    """Process one row for raw ISI — for parallel execution."""
    record = {
        'animal': row['animal'],
        'geno_label': 'AD' if row['geno'] == 'app_ps1' else 'WT',
        'age': row['age'], 'age_group': row['age_group'],
        'brain_region': row['brain_region'], 'cell_type': row['cell_type'],
        'peak_latency': row['peak_latency'],
        'label': 1 if row['geno'] == 'app_ps1' else 0
    }
    for state in ['wake', 'nrem', 'rem']:
        hist = isi_to_histogram(row['{}_isi'.format(state)])
        for b in range(len(hist)):
            record['{}_bin{}'.format(state, b)] = hist[b]
    return record


print('Converting ISI arrays to {}-bin log-spaced histograms...'.format(N_BINS))
print('Bin range: {:.1f}ms to {:.1f}s'.format(BIN_EDGES[0] * 1000, BIN_EDGES[-1]))

t0 = time.time()
rows_list = [row for _, row in df.iterrows()]
raw_isi_records = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(_process_raw_isi_row)(row) for row in rows_list
)
print('Histogram extraction: {:.1f}s'.format(time.time() - t0))

raw_isi_df = pd.DataFrame(raw_isi_records)
raw_isi_feat_cols = [c for c in raw_isi_df.columns if c not in meta_cols]
print('Raw ISI features: {} ({} bins x 3 states)'.format(
    len(raw_isi_feat_cols), N_BINS))


In [ ]:
# =============================================================================
# PART 1: RUN RAW ISI MLP — ALL, OLD, YOUNG
# =============================================================================
master_results['Raw ISI'] = run_all_three_splits(
    raw_isi_df, raw_isi_feat_cols,
    approach_name='Raw ISI Histograms',
    hidden_layers=(256, 128, 64)
)

In [ ]:
# Part 1 Visualization
plot_results_grid(master_results['Raw ISI'], 'Part 1: Raw ISI Histogram MLP')

---
# Part 2: ISI Feature MLP
Extract hand-crafted statistical features from ISI distributions (mean, CV, skew, burst fraction, etc.),
then classify with an MLP.

In [ ]:
# =============================================================================
# ISI FEATURE EXTRACTION (PARALLELIZED)
# =============================================================================
print('=' * 70)
print('PART 2: EXTRACTING ISI DISTRIBUTION FEATURES')
print('=' * 70)


def extract_isi_features(isis, prefix):
    """Extract comprehensive distribution features from an ISI array."""
    feats = {}
    if len(isis) < 50:
        keys = ['mean', 'median', 'std', 'cv', 'skew', 'kurt', 'fano',
                'q5', 'q10', 'q25', 'q75', 'q90', 'q95', 'iqr',
                'burst_frac', 'pause_frac', 'slow_frac',
                'log_mean', 'log_std', 'log_skew',
                'mode_isi', 'entropy', 'firing_rate']
        for k in keys:
            feats['{}_{}'.format(prefix, k)] = np.nan
        return feats

    m = np.mean(isis)
    feats['{}_mean'.format(prefix)] = m
    feats['{}_median'.format(prefix)] = np.median(isis)
    feats['{}_std'.format(prefix)] = np.std(isis)
    feats['{}_cv'.format(prefix)] = np.std(isis) / m if m > 0 else np.nan
    feats['{}_skew'.format(prefix)] = stats.skew(isis)
    feats['{}_kurt'.format(prefix)] = stats.kurtosis(isis)
    feats['{}_fano'.format(prefix)] = np.var(isis) / m if m > 0 else np.nan

    for q in [5, 10, 25, 75, 90, 95]:
        feats['{}_q{}'.format(prefix, q)] = np.percentile(isis, q)
    feats['{}_iqr'.format(prefix)] = feats['{}_q75'.format(prefix)] - feats['{}_q25'.format(prefix)]

    feats['{}_burst_frac'.format(prefix)] = np.mean(isis < 0.010)
    feats['{}_pause_frac'.format(prefix)] = np.mean(isis > 0.5)
    feats['{}_slow_frac'.format(prefix)] = np.mean(isis > 1.0)

    log_isis = np.log10(isis[isis > 0] + 1e-6)
    feats['{}_log_mean'.format(prefix)] = np.mean(log_isis)
    feats['{}_log_std'.format(prefix)] = np.std(log_isis)
    feats['{}_log_skew'.format(prefix)] = stats.skew(log_isis)

    try:
        hist, bin_edges = np.histogram(isis, bins=100, range=(0, min(5, np.percentile(isis, 99))))
        feats['{}_mode_isi'.format(prefix)] = bin_edges[np.argmax(hist)]
        hist_norm = hist / hist.sum()
        hist_norm = hist_norm[hist_norm > 0]
        feats['{}_entropy'.format(prefix)] = -np.sum(hist_norm * np.log2(hist_norm))
    except Exception:
        feats['{}_mode_isi'.format(prefix)] = np.nan
        feats['{}_entropy'.format(prefix)] = np.nan

    feats['{}_firing_rate'.format(prefix)] = 1.0 / m if m > 0 else np.nan
    return feats


def extract_epoch_dynamics(isi_list, prefix):
    """Extract epoch-to-epoch dynamics features."""
    feats = {}
    epoch_means = []
    epoch_cvs = []
    epoch_counts = []
    for arr in isi_list:
        if isinstance(arr, np.ndarray):
            valid = arr[(arr > 0) & (arr <= 60)]
            if len(valid) > 10:
                epoch_means.append(np.mean(valid))
                epoch_cvs.append(np.std(valid) / np.mean(valid) if np.mean(valid) > 0 else np.nan)
                epoch_counts.append(len(valid))

    feats['{}_n_epochs'.format(prefix)] = len(epoch_means)
    if len(epoch_means) > 1:
        feats['{}_epoch_mean_cv'.format(prefix)] = np.std(epoch_means) / np.mean(epoch_means)
        feats['{}_epoch_mean_range'.format(prefix)] = np.max(epoch_means) - np.min(epoch_means)
        feats['{}_epoch_cv_mean'.format(prefix)] = np.nanmean(epoch_cvs)
        feats['{}_epoch_count_cv'.format(prefix)] = np.std(epoch_counts) / np.mean(epoch_counts)
    else:
        for k in ['epoch_mean_cv', 'epoch_mean_range', 'epoch_cv_mean', 'epoch_count_cv']:
            feats['{}_{}'.format(prefix, k)] = np.nan
    if len(epoch_means) > 2:
        feats['{}_epoch_autocorr'.format(prefix)] = np.corrcoef(epoch_means[:-1], epoch_means[1:])[0, 1]
    else:
        feats['{}_epoch_autocorr'.format(prefix)] = np.nan
    return feats


def _process_isi_features_row(row):
    """Process one row for ISI features — for parallel execution."""
    feats = {
        'animal': row['animal'],
        'geno_label': 'AD' if row['geno'] == 'app_ps1' else 'WT',
        'age': row['age'], 'age_group': row['age_group'],
        'brain_region': row['brain_region'], 'cell_type': row['cell_type'],
        'peak_latency': row['peak_latency'],
        'label': 1 if row['geno'] == 'app_ps1' else 0
    }
    for state in ['wake', 'nrem', 'rem']:
        isis = clean_isi_array(row['{}_isi'.format(state)])
        feats.update(extract_isi_features(isis, state))
        feats.update(extract_epoch_dynamics(row['{}_isi'.format(state)], state))

    for rname, num, den in [('rem_wake_rate_ratio', 'rem_firing_rate', 'wake_firing_rate'),
                            ('rem_nrem_rate_ratio', 'rem_firing_rate', 'nrem_firing_rate'),
                            ('nrem_wake_rate_ratio', 'nrem_firing_rate', 'wake_firing_rate'),
                            ('rem_wake_cv_ratio', 'rem_cv', 'wake_cv'),
                            ('rem_nrem_cv_ratio', 'rem_cv', 'nrem_cv')]:
        n_val = feats.get(num)
        d_val = feats.get(den)
        if (n_val is not None and d_val is not None
                and not np.isnan(n_val) and not np.isnan(d_val) and d_val > 0):
            feats[rname] = n_val / d_val
        else:
            feats[rname] = np.nan
    return feats


print('Extracting ISI features for {:,} neurons using {} cores...'.format(len(df), N_JOBS))
t0 = time.time()
# Reuse rows_list from Part 1 if available, otherwise rebuild
try:
    _ = rows_list[0]
except NameError:
    rows_list = [row for _, row in df.iterrows()]

isi_feature_list = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(_process_isi_features_row)(row) for row in rows_list
)
print('ISI feature extraction: {:.1f}s'.format(time.time() - t0))

isi_df = pd.DataFrame(isi_feature_list)
isi_feat_cols = [c for c in isi_df.columns if c not in meta_cols]
print('Extracted {} ISI features per neuron'.format(len(isi_feat_cols)))


In [ ]:
# =============================================================================
# PART 2: RUN ISI FEATURE MLP — ALL, OLD, YOUNG
# =============================================================================
master_results['ISI Features'] = run_all_three_splits(
    isi_df, isi_feat_cols,
    approach_name='ISI Features',
    hidden_layers=(64, 32)
)

In [ ]:
# Part 2 Visualization
plot_results_grid(master_results['ISI Features'], 'Part 2: ISI Feature MLP')

In [ ]:
# =============================================================================
# ISI FEATURE IMPORTANCE (PARALLELIZED)
# =============================================================================
print('=' * 70)
print('ISI FEATURE IMPORTANCE (Permutation-Based)')
print('=' * 70)

X_all = isi_df[isi_feat_cols].values.astype(float)
y_all = isi_df['label'].values
for col_idx in range(X_all.shape[1]):
    nan_mask = np.isnan(X_all[:, col_idx])
    if nan_mask.any():
        X_all[nan_mask, col_idx] = np.nanmedian(X_all[:, col_idx])

scaler_f = StandardScaler()
X_sc = scaler_f.fit_transform(X_all)
mlp_f = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                      solver='adam', alpha=0.01, max_iter=300,
                      early_stopping=True, validation_fraction=0.15, random_state=42)
mlp_f.fit(X_sc, y_all)

t0 = time.time()
perm_imp = permutation_importance(mlp_f, X_sc, y_all, n_repeats=10,
                                  random_state=42, scoring='roc_auc',
                                  n_jobs=N_JOBS)
print('Permutation importance: {:.1f}s'.format(time.time() - t0))

imp_df = pd.DataFrame({
    'feature': isi_feat_cols,
    'importance': perm_imp.importances_mean,
    'std': perm_imp.importances_std
}).sort_values('importance', ascending=False)

print('\nTop 20 ISI Features:')
for i, (_, r) in enumerate(imp_df.head(20).iterrows(), 1):
    print('{:2d}. {:<38} {:.4f} +/- {:.4f}'.format(i, r['feature'], r['importance'], r['std']))

fig, ax = plt.subplots(figsize=(10, 8))
top20 = imp_df.head(20)
colors = []
for f in top20['feature']:
    if f.startswith('rem_'): colors.append('crimson')
    elif f.startswith('nrem_'): colors.append('navy')
    elif f.startswith('wake_'): colors.append('goldenrod')
    else: colors.append('gray')
ax.barh(range(len(top20)), top20['importance'].values, xerr=top20['std'].values,
        color=colors, edgecolor='black', capsize=3)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Permutation Importance (AUC decrease)')
ax.set_title('Top 20 ISI Features\n(Red=REM, Blue=NREM, Gold=Wake)', fontweight='bold')
plt.tight_layout()
plt.show()


---
# Part 3: Behavioral & Sleep Architecture Feature MLP
Classify using features that capture **behavioral state dynamics**:
- Proportion of time in REM/NREM/Wake
- Average epoch activity per state (proxy for bout length)
- NREM→REM transition dynamics
- Circadian phase alignment
- State-dependent firing rate modulation

In [ ]:
# =============================================================================
# BEHAVIORAL FEATURE EXTRACTION (PARALLELIZED)
# =============================================================================
print('=' * 70)
print('PART 3: EXTRACTING BEHAVIORAL & SLEEP ARCHITECTURE FEATURES')
print('=' * 70)


def parse_epoch_timing(epoch_str):
    """Parse epoch string like '0914_0_12' -> (month, day, start_hour, end_hour)."""
    try:
        parts = epoch_str.split('_')
        date_part = parts[0]
        return int(date_part[:2]), int(date_part[2:]), int(parts[1]), int(parts[2])
    except Exception:
        return None, None, None, None


def extract_behavioral_features(row):
    """Extract features about sleep/wake behavior and circadian patterns."""
    feats = {
        'animal': row['animal'],
        'geno_label': 'AD' if row['geno'] == 'app_ps1' else 'WT',
        'age': row['age'], 'age_group': row['age_group'],
        'brain_region': row['brain_region'], 'cell_type': row['cell_type'],
        'peak_latency': row['peak_latency'],
        'label': 1 if row['geno'] == 'app_ps1' else 0
    }

    state_epoch_counts = {}
    state_total_spikes = {}
    state_epoch_sizes = {}

    for state in ['wake', 'nrem', 'rem']:
        n_epochs = 0
        total_spikes = 0
        epoch_sizes = []
        for arr in row['{}_isi'.format(state)]:
            if isinstance(arr, np.ndarray) and len(arr) > 10:
                n_epochs += 1
                total_spikes += len(arr)
                epoch_sizes.append(len(arr))
        state_epoch_counts[state] = n_epochs
        state_total_spikes[state] = total_spikes
        state_epoch_sizes[state] = epoch_sizes

    total_epochs = sum(state_epoch_counts.values())
    total_spikes = sum(state_total_spikes.values())

    for state in ['wake', 'nrem', 'rem']:
        feats['{}_frac'.format(state)] = state_epoch_counts[state] / total_epochs if total_epochs > 0 else 0
    feats['rem_nrem_ratio'] = state_epoch_counts['rem'] / state_epoch_counts['nrem'] if state_epoch_counts['nrem'] > 0 else 0
    feats['rem_wake_ratio'] = state_epoch_counts['rem'] / state_epoch_counts['wake'] if state_epoch_counts['wake'] > 0 else 0

    for state in ['wake', 'nrem', 'rem']:
        sizes = state_epoch_sizes[state]
        if len(sizes) > 0:
            feats['{}_avg_epoch_spikes'.format(state)] = np.mean(sizes)
            feats['{}_std_epoch_spikes'.format(state)] = np.std(sizes)
            feats['{}_max_epoch_spikes'.format(state)] = np.max(sizes)
            feats['{}_min_epoch_spikes'.format(state)] = np.min(sizes)
        else:
            for k in ['avg', 'std', 'max', 'min']:
                feats['{}_{}_epoch_spikes'.format(state, k)] = 0

    state_rates = {}
    for state in ['wake', 'nrem', 'rem']:
        isis = clean_isi_array(row['{}_isi'.format(state)])
        state_rates[state] = 1.0 / np.mean(isis) if len(isis) > 50 and np.mean(isis) > 0 else np.nan

    for label, s1, s2 in [('wake_to_nrem_rate_change', 'nrem', 'wake'),
                          ('nrem_to_rem_rate_change', 'rem', 'nrem'),
                          ('wake_to_rem_rate_change', 'rem', 'wake')]:
        r1 = state_rates.get(s1, np.nan)
        r2 = state_rates.get(s2, np.nan)
        if not np.isnan(r1) and not np.isnan(r2) and r2 > 0:
            feats[label] = r1 / r2
        else:
            feats[label] = np.nan

    valid_rates = [r for r in state_rates.values() if not np.isnan(r)]
    if len(valid_rates) >= 2:
        feats['state_modulation_index'] = np.std(valid_rates) / np.mean(valid_rates)
        feats['state_rate_range'] = np.max(valid_rates) / np.min(valid_rates) if np.min(valid_rates) > 0 else np.nan
    else:
        feats['state_modulation_index'] = np.nan
        feats['state_rate_range'] = np.nan

    _, _, start_hr, _ = parse_epoch_timing(row['epoch'])
    if start_hr is not None:
        feats['recording_start_hour'] = start_hr
        feats['is_light_phase'] = 1 if (7 <= start_hr < 19) else 0
        feats['is_dark_phase'] = 1 - feats['is_light_phase']
    else:
        feats['recording_start_hour'] = np.nan
        feats['is_light_phase'] = np.nan
        feats['is_dark_phase'] = np.nan

    feats['rem_to_nrem_spike_ratio'] = (state_total_spikes['rem'] / state_total_spikes['nrem']
                                        if state_total_spikes['nrem'] > 0 else 0)

    for state in ['wake', 'nrem', 'rem']:
        sizes = state_epoch_sizes[state]
        if len(sizes) > 1:
            sizes_arr = np.array(sizes, dtype=float)
            sizes_norm = sizes_arr / sizes_arr.sum()
            feats['{}_spike_entropy'.format(state)] = -np.sum(sizes_norm * np.log2(sizes_norm + 1e-10))
            sorted_s = np.sort(sizes_arr)
            n = len(sorted_s)
            feats['{}_gini'.format(state)] = (2 * np.sum((np.arange(1, n+1) * sorted_s)) / (n * np.sum(sorted_s)) - (n + 1) / n)
        else:
            feats['{}_spike_entropy'.format(state)] = 0
            feats['{}_gini'.format(state)] = 0

    feats['total_epochs'] = total_epochs
    feats['total_spikes'] = total_spikes
    return feats


print('Extracting behavioral features using {} cores...'.format(N_JOBS))
t0 = time.time()
behav_feature_list = Parallel(n_jobs=N_JOBS, verbose=1)(
    delayed(extract_behavioral_features)(row) for row in rows_list
)
print('Behavioral feature extraction: {:.1f}s'.format(time.time() - t0))

behav_df = pd.DataFrame(behav_feature_list)
behav_feat_cols = [c for c in behav_df.columns if c not in meta_cols]
print('Extracted {} behavioral features per neuron'.format(len(behav_feat_cols)))


In [ ]:
# =============================================================================
# BEHAVIORAL FEATURE EXPLORATION
# =============================================================================
key_behav_feats = ['rem_frac', 'nrem_frac', 'wake_frac', 'rem_nrem_ratio',
                   'wake_to_nrem_rate_change', 'nrem_to_rem_rate_change',
                   'state_modulation_index', 'rem_to_nrem_spike_ratio',
                   'rem_avg_epoch_spikes', 'rem_spike_entropy']
key_behav_feats = [f for f in key_behav_feats if f in behav_df.columns]

behav_df['group'] = behav_df['geno_label'] + '_' + behav_df['age_group']
order = ['WT_YOUNG', 'AD_YOUNG', 'WT_OLD', 'AD_OLD']
palette = {'WT_YOUNG': 'steelblue', 'AD_YOUNG': 'lightcoral',
           'WT_OLD': 'darkblue', 'AD_OLD': 'darkred'}

n_feats = len(key_behav_feats)
n_cols = 3
n_rows = (n_feats + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(key_behav_feats):
    ax = axes[i]
    data = behav_df[behav_df[feat].notna()]
    sns.boxplot(data=data, x='group', y=feat, order=order,
                palette=palette, ax=ax, showfliers=False)
    ax.set_title(feat, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    wt_old = data[(data['geno_label'] == 'WT') & (data['age_group'] == 'OLD')][feat].dropna()
    ad_old = data[(data['geno_label'] == 'AD') & (data['age_group'] == 'OLD')][feat].dropna()
    if len(wt_old) > 10 and len(ad_old) > 10:
        _, p = stats.mannwhitneyu(wt_old, ad_old)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ax.set_xlabel('OLD WT vs AD: p={:.1e} ({})'.format(p, sig), fontsize=9)
for i in range(n_feats, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Key Behavioral Features by Group', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# PART 3: RUN BEHAVIORAL MLP — ALL, OLD, YOUNG
# =============================================================================
master_results['Behavioral'] = run_all_three_splits(
    behav_df, behav_feat_cols,
    approach_name='Behavioral Features',
    hidden_layers=(32, 16)
)

In [ ]:
# Part 3 Visualization
plot_results_grid(master_results['Behavioral'], 'Part 3: Behavioral Feature MLP')

---
# Part 4: All-Dimensions Combined MLP
Combine ISI features + behavioral features + metadata for maximum classification power.

In [ ]:
# =============================================================================
# MERGE ALL FEATURES
# =============================================================================
print('=' * 70)
print('PART 4: BUILDING COMBINED FEATURE SET')
print('=' * 70)

combined_df = isi_df[meta_cols].copy()

# Add ISI features
for col in isi_feat_cols:
    combined_df[col] = isi_df[col].values

# Add behavioral features (skip overlapping)
for col in behav_feat_cols:
    if col not in combined_df.columns:
        combined_df[col] = behav_df[col].values

# Add encoded metadata
combined_df['brain_region_enc'] = LabelEncoder().fit_transform(combined_df['brain_region'])
combined_df['cell_type_enc'] = LabelEncoder().fit_transform(combined_df['cell_type'])

all_feat_cols = isi_feat_cols + [c for c in behav_feat_cols if c not in isi_feat_cols]
all_feat_cols += ['brain_region_enc', 'cell_type_enc', 'peak_latency']
all_feat_cols = list(dict.fromkeys(all_feat_cols))  # deduplicate

print('Combined feature set: {} features'.format(len(all_feat_cols)))
print('  ISI: {}  |  Behavioral: {}  |  Metadata: 3'.format(
    len(isi_feat_cols), len(behav_feat_cols)))

In [ ]:
# =============================================================================
# PART 4: RUN COMBINED MLP — ALL, OLD, YOUNG
# =============================================================================
master_results['Combined'] = run_all_three_splits(
    combined_df, all_feat_cols,
    approach_name='All Features Combined',
    hidden_layers=(256, 128, 64, 32)
)

In [ ]:
# Part 4 Visualization
plot_results_grid(master_results['Combined'], 'Part 4: All-Features Combined MLP')

In [ ]:
# =============================================================================
# COMBINED FEATURE IMPORTANCE (PARALLELIZED)
# =============================================================================
print('=' * 70)
print('COMBINED MODEL: FEATURE IMPORTANCE')
print('=' * 70)

X_comb = combined_df[all_feat_cols].values.astype(float)
y_comb = combined_df['label'].values
for col_idx in range(X_comb.shape[1]):
    nan_mask = np.isnan(X_comb[:, col_idx])
    if nan_mask.any():
        med = np.nanmedian(X_comb[:, col_idx])
        X_comb[nan_mask, col_idx] = med if not np.isnan(med) else 0

scaler_c = StandardScaler()
X_cs = scaler_c.fit_transform(X_comb)
mlp_c = MLPClassifier(hidden_layer_sizes=(256, 128, 64, 32), activation='relu',
                      solver='adam', alpha=0.01, max_iter=300,
                      early_stopping=True, validation_fraction=0.15, random_state=42)
mlp_c.fit(X_cs, y_comb)

t0 = time.time()
perm_imp_c = permutation_importance(mlp_c, X_cs, y_comb, n_repeats=10,
                                    random_state=42, scoring='roc_auc',
                                    n_jobs=N_JOBS)
print('Permutation importance: {:.1f}s'.format(time.time() - t0))

imp_comb_df = pd.DataFrame({
    'feature': all_feat_cols,
    'importance': perm_imp_c.importances_mean,
    'std': perm_imp_c.importances_std
}).sort_values('importance', ascending=False)

def categorize_feature(f):
    if f in behav_feat_cols and f not in isi_feat_cols:
        return 'Behavioral'
    elif f in isi_feat_cols and f not in behav_feat_cols:
        return 'ISI'
    elif f in ['brain_region_enc', 'cell_type_enc', 'age_norm', 'peak_latency']:
        return 'Metadata'
    return 'Shared'

imp_comb_df['category'] = imp_comb_df['feature'].apply(categorize_feature)

print('\nTop 25 Features (Combined Model):')
for i, (_, r) in enumerate(imp_comb_df.head(25).iterrows(), 1):
    print('{:2d}. {:<38} {:.4f} +/- {:.4f}  [{}]'.format(
        i, r['feature'], r['importance'], r['std'], r['category']))

fig, ax = plt.subplots(figsize=(10, 10))
top25 = imp_comb_df.head(25)
cat_colors = {'ISI': 'steelblue', 'Behavioral': 'crimson',
              'Metadata': 'forestgreen', 'Shared': 'gray'}
colors = [cat_colors[c] for c in top25['category']]
ax.barh(range(len(top25)), top25['importance'].values, xerr=top25['std'].values,
        color=colors, edgecolor='black', capsize=3)
ax.set_yticks(range(len(top25)))
ax.set_yticklabels(top25['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Permutation Importance (AUC decrease)')
ax.set_title('Top 25 Features - Combined Model\n(Blue=ISI, Red=Behavioral, Green=Metadata)',
             fontweight='bold')
plt.tight_layout()
plt.show()


---
# Head-to-Head Comparison: All Approaches x All Splits

In [ ]:
# =============================================================================
# MASTER COMPARISON TABLE
# =============================================================================
print('=' * 90)
print('MASTER COMPARISON: ALL APPROACHES x ALL SPLITS')
print('=' * 90)
print()
print('{:<25} {:<10} {:<12} {:<12} {:<12} {:<12}'.format(
    'Approach', 'Split', 'N_Acc', 'N_AUC', 'A_Acc', 'A_AUC'))
print('-' * 85)

comparison_rows = []
for approach_name in ['Raw ISI', 'ISI Features', 'Behavioral', 'Combined']:
    approach_res = master_results[approach_name]
    for split in ['all', 'old', 'young']:
        if split in approach_res:
            r = approach_res[split]
            print('{:<25} {:<10} {:<12.1%} {:<12.3f} {:<12.1%} {:<12.3f}'.format(
                approach_name, split.upper(),
                r['neuron_acc'], r['neuron_auc'],
                r['animal_acc'], r['animal_auc']))
            comparison_rows.append({
                'approach': approach_name, 'split': split,
                'neuron_auc': r['neuron_auc'], 'animal_auc': r['animal_auc']
            })
    print()  # blank line between approaches

# Find best per split
print('\nBEST APPROACH PER SPLIT (by Animal AUC):')
print('-' * 60)
comp_df = pd.DataFrame(comparison_rows)
for split in ['all', 'old', 'young']:
    split_df = comp_df[comp_df['split'] == split]
    if len(split_df) > 0:
        best = split_df.loc[split_df['animal_auc'].idxmax()]
        print('  {}: {} (Animal AUC = {:.3f})'.format(
            split.upper(), best['approach'], best['animal_auc']))

In [ ]:
# =============================================================================
# COMPARISON VISUALIZATIONS
# =============================================================================

# ROC comparison for ALL animals split
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
split_labels = {'all': 'All Animals', 'old': 'OLD Only (>212d)', 'young': 'YOUNG Only (<=212d)'}
approach_colors = {'Raw ISI': 'purple', 'ISI Features': 'steelblue',
                   'Behavioral': 'crimson', 'Combined': 'forestgreen'}

for col_idx, split in enumerate(['all', 'old', 'young']):
    ax = axes[col_idx]
    for approach_name, approach_res in master_results.items():
        if split in approach_res:
            r = approach_res[split]
            fpr, tpr, _ = roc_curve(r['y_true'], r['y_probs'])
            ax.plot(fpr, tpr, color=approach_colors[approach_name], lw=2,
                    label='{} ({:.3f})'.format(approach_name, r['neuron_auc']))
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.set_title('{}: Neuron-Level ROC'.format(split_labels[split]), fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('ROC Comparison Across All Approaches', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, metric in enumerate(['neuron_auc', 'animal_auc']):
    ax = axes[ax_idx]
    approaches = list(master_results.keys())
    splits = ['all', 'old', 'young']
    x = np.arange(len(approaches))
    width = 0.25
    split_colors = {'all': 'gray', 'old': 'navy', 'young': 'coral'}

    for i, split in enumerate(splits):
        vals = []
        for approach in approaches:
            if split in master_results[approach]:
                vals.append(master_results[approach][split][metric])
            else:
                vals.append(0)
        ax.bar(x + i * width, vals, width, label=split.upper(),
               color=split_colors[split], edgecolor='black')

    ax.set_xticks(x + width)
    ax.set_xticklabels(approaches, rotation=15)
    ax.set_ylabel('AUC')
    ax.set_title(metric.replace('_', ' ').title(), fontweight='bold')
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='Chance')

plt.suptitle('AUC Comparison: Approaches x Age Splits', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# BRAIN REGION ANALYSIS (using best overall approach)
# =============================================================================
print('=' * 70)
print('BRAIN REGION ANALYSIS: Which region is most informative?')
print('=' * 70)

region_results = {}
for region in ['ACad', 'CA1', 'RSPv', 'V1']:
    print('\n--- {} ---'.format(region))
    subset = combined_df[combined_df['brain_region'] == region].copy()
    n_animals = subset['animal'].nunique()
    n_ad = subset[subset['label'] == 1]['animal'].nunique()
    n_wt = subset[subset['label'] == 0]['animal'].nunique()
    print('  {} animals ({} WT, {} AD), {:,} neurons'.format(
        n_animals, n_wt, n_ad, len(subset)))
    if n_ad < 2 or n_wt < 2:
        print('  SKIPPED')
        continue
    region_results[region] = run_mlp_classification(
        subset, all_feat_cols,
        title=region, hidden_layers=(32, 16))

print('\n' + '=' * 70)
print('REGION COMPARISON SUMMARY')
print('=' * 70)
print('{:<10} {:<12} {:<12} {:<12} {:<12}'.format(
    'Region', 'N_Acc', 'N_AUC', 'A_Acc', 'A_AUC'))
print('-' * 60)
for region, res in region_results.items():
    print('{:<10} {:<12.1%} {:<12.3f} {:<12.1%} {:<12.3f}'.format(
        region, res['neuron_acc'], res['neuron_auc'],
        res['animal_acc'], res['animal_auc']))

---
# Summary & Conclusions

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================
print('=' * 90)
print('FINAL SUMMARY')
print('=' * 90)
print()
print('FULL RESULTS TABLE (Leave-One-Animal-Out CV)')
print('-' * 85)
print('{:<25} {:<10} {:<12} {:<12} {:<12} {:<12}'.format(
    'Approach', 'Split', 'N_Acc', 'N_AUC', 'A_Acc', 'A_AUC'))
print('-' * 85)
for approach_name in ['Raw ISI', 'ISI Features', 'Behavioral', 'Combined']:
    approach_res = master_results[approach_name]
    for split in ['all', 'old', 'young']:
        if split in approach_res:
            r = approach_res[split]
            print('{:<25} {:<10} {:<12.1%} {:<12.3f} {:<12.1%} {:<12.3f}'.format(
                approach_name, split.upper(),
                r['neuron_acc'], r['neuron_auc'],
                r['animal_acc'], r['animal_auc']))
    print()

print('\nKEY INSIGHTS:')
print('-' * 85)

# Best overall
best_all = max(
    [(name, master_results[name]['all']) for name in master_results if 'all' in master_results[name]],
    key=lambda x: x[1]['animal_auc']
)
print('1. Best overall approach: {} (Animal AUC = {:.3f})'.format(
    best_all[0], best_all[1]['animal_auc']))

# OLD vs YOUNG
for split_name in ['old', 'young']:
    candidates = [(name, master_results[name][split_name])
                  for name in master_results if split_name in master_results[name]]
    if candidates:
        best = max(candidates, key=lambda x: x[1]['animal_auc'])
        print('   Best for {}: {} (Animal AUC = {:.3f})'.format(
            split_name.upper(), best[0], best[1]['animal_auc']))

# Raw ISI vs features
if 'all' in master_results['Raw ISI'] and 'all' in master_results['ISI Features']:
    raw_auc = master_results['Raw ISI']['all']['animal_auc']
    feat_auc = master_results['ISI Features']['all']['animal_auc']
    if raw_auc > feat_auc:
        print('2. Raw ISI ({:.3f}) > ISI Features ({:.3f})'.format(raw_auc, feat_auc))
        print('   -> MLP can learn better representations than hand-crafted features')
    else:
        print('2. ISI Features ({:.3f}) > Raw ISI ({:.3f})'.format(feat_auc, raw_auc))
        print('   -> Hand-crafted features provide better signal than raw histograms')

# ISI vs Behavioral
if 'all' in master_results['ISI Features'] and 'all' in master_results['Behavioral']:
    isi_a = master_results['ISI Features']['all']['animal_auc']
    beh_a = master_results['Behavioral']['all']['animal_auc']
    if isi_a > beh_a:
        print('3. ISI ({:.3f}) > Behavioral ({:.3f})'.format(isi_a, beh_a))
        print('   -> Spike timing distributions carry more AD-discriminative information')
    else:
        print('3. Behavioral ({:.3f}) > ISI ({:.3f})'.format(beh_a, isi_a))
        print('   -> Sleep architecture changes are more diagnostic')

# OLD vs YOUNG separation
print('\n4. OLD vs YOUNG classification quality:')
for approach_name in ['Raw ISI', 'ISI Features', 'Behavioral', 'Combined']:
    old_auc = master_results[approach_name].get('old', {}).get('animal_auc', None)
    young_auc = master_results[approach_name].get('young', {}).get('animal_auc', None)
    if old_auc is not None and young_auc is not None:
        print('   {}: OLD AUC={:.3f}, YOUNG AUC={:.3f} (diff={:+.3f})'.format(
            approach_name, old_auc, young_auc, old_auc - young_auc))

print()
print('METHODOLOGY:')
print('- Leave-One-Animal-Out CV (no data leakage)')
print('- Animal prediction = majority vote across all neurons')
print('- MLP: ReLU, Adam, L2=0.01, early stopping, adaptive LR')
print('- Age cutoff: {} days (APP/PS1 symptom onset)'.format(AGE_CUTOFF))
print('- OLD: >{}d | YOUNG: <={}d'.format(AGE_CUTOFF, AGE_CUTOFF))